# Ultimate NIDS Pipeline: Hybrid Sniper Ensemble (PyTorch + XGBoost)
**Goal:** Fix the "R2L Blind Spot" without causing False Alarms on Probes.

**The "Creative" Approach: The Sniper Architecture**
Neural Networks (ResNet) smooth out decision boundaries, often missing the sharp, specific rules that define R2L attacks. We fix this by adding a parallel "Sniper" model.

**Architecture:**
1.  **Vector Space (ResNet):** A Deep Metric Learning model handles the "Macro" view (Normal, DoS, Probe).
2.  **The "Sniper" (XGBoost):** A specialized binary classifier trained specifically to hunt R2L and U2R attacks using raw features. It captures the exact signatures the Neural Net misses.
3.  **Cascade Inference:** We prioritize the Sniper. If it detects an R2L/U2R signature, we override the ResNet. Otherwise, we trust the ResNet's general classification.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc

# The Hybrid Stack
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Processing Unit: {device}")

# Config
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

Processing Unit: cuda


## 1. Data Loading

In [2]:
DATA_DIR = 'Data'
CSV_FILE = os.path.join(DATA_DIR, 'network_connections.csv')
MAP_FILE = os.path.join(DATA_DIR, 'attack2category_map.txt')

# 1. Load Mapping
attack_map = {'normal': 'normal'}
try:
    with open(MAP_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attack_map[parts[0]] = parts[1]
except FileNotFoundError:
    print("Warning: Map file not found. Creating dummy map.")

# 2. Load Data
df = pd.read_csv(CSV_FILE)
df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
df['category'] = df['label'].map(attack_map).fillna('other')
df.drop_duplicates(inplace=True)
print(f"Data Loaded. Shape: {df.shape}")

Data Loaded. Shape: (125973, 43)


## 2. Feature Engineering (Dual Pipeline)
We keep raw features for the Sniper (XGBoost loves raw flags) and Vector features for the ResNet.

In [3]:
def prepare_features(data, fit=False, encoders=None):
    df_eng = data.copy()
    
    # 1. Log Transform
    for col in ['src_bytes', 'dst_bytes', 'duration']:
        if col in df_eng.columns:
            df_eng[col] = np.log1p(df_eng[col]).astype(np.float32)

    # 2. R2L Specialized Feature: System Access Risk
    risk_cols = ['num_failed_logins', 'is_guest_login', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations']
    df_eng['system_access_risk'] = 0.0
    for col in risk_cols:
        if col in df_eng.columns:
            df_eng['system_access_risk'] += df_eng[col].astype(np.float32)
            
    # 3. Frequency Encoding
    cat_cols = ['protocol_type', 'service', 'flag']
    new_encoders = {}
    for col in cat_cols:
        if fit:
            freq_map = df_eng[col].value_counts(normalize=True).to_dict()
            df_eng[col] = df_eng[col].map(freq_map).astype(np.float32)
            new_encoders[col] = freq_map
        else:
            df_eng[col] = df_eng[col].map(encoders[col]).fillna(0).astype(np.float32)
            
    return df_eng, new_encoders

X = df.drop(['label', 'category'], axis=1)
y = df['category']

X_eng, encoders = prepare_features(X, fit=True)

# Save encoder for final decoding
le_y = LabelEncoder()
y_vec = le_y.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_eng, y_vec, test_size=0.2, stratify=y_vec, random_state=42)
print("Data Prepared.")

Data Prepared.


## 3. Training the "Sniper" (XGBoost for Rare Attacks)
This model's ONLY job is to find R2L and U2R. It ignores everything else.

In [4]:
print("--- Training R2L/U2R Sniper ---")

# 1. Create Binary Target: 1 for R2L/U2R, 0 for Everything Else
# We need to know which integers correspond to r2l/u2r
target_classes = le_y.transform(['r2l', 'u2r'])
y_train_sniper = np.isin(y_train, target_classes).astype(int)

# 2. Aggressive Balancing for Sniper
# We downsample Normal/DoS so the Sniper sees a 50/50 split of Rare vs Common
train_df_sniper = X_train.copy()
train_df_sniper['target'] = y_train_sniper

rare_df = train_df_sniper[train_df_sniper['target'] == 1]
common_df = train_df_sniper[train_df_sniper['target'] == 0]

# Boost Rare x10, Downsample Common to match
rare_boosted = pd.concat([rare_df] * 10, ignore_index=True)
common_downsampled = common_df.sample(n=len(rare_boosted), random_state=42)

sniper_data = pd.concat([rare_boosted, common_downsampled]).sample(frac=1, random_state=42)
print(f"Sniper Training Data: {sniper_data.shape} (Balanced 50/50 Rare/Common)")

# 3. Train XGBoost Sniper
sniper = XGBClassifier(
    n_estimators=200,
    max_depth=10,        # Deep trees to find specific signatures
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=1.5 # Slight bias towards Positive (Rare)
)
sniper.fit(sniper_data.drop('target', axis=1), sniper_data['target'])
print("Sniper Trained.")

--- Training R2L/U2R Sniper ---
Sniper Training Data: (16760, 43) (Balanced 50/50 Rare/Common)
Sniper Trained.


## 4. Vector Space Engineering (For the Generalist ResNet)
We still use the robust vector space for the main classification task.

In [5]:
print("--- Engineering Vector Space (ResNet) ---")

# 1. Re-balance Main Training Data (Moderate balancing for ResNet)
train_df = X_train.copy()
train_df['target'] = y_train
dfs = []
for cls in np.unique(y_train):
    cls_df = train_df[train_df['target'] == cls]
    if len(cls_df) < 2000: 
        dfs.append(pd.concat([cls_df] * (2000 // len(cls_df) + 1)).sample(n=2000))
    else:
        dfs.append(cls_df.sample(n=min(len(cls_df), 10000)))
train_bal = pd.concat(dfs).sample(frac=1)
X_train_bal = train_bal.drop('target', axis=1)
y_train_bal = train_bal['target']

# 2. Scale & Project
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train_bal).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)

nca = NeighborhoodComponentsAnalysis(n_components=15, init='pca', random_state=42)
idx = np.random.choice(len(X_train_sc), size=min(3000, len(X_train_sc)), replace=False)
nca.fit(X_train_sc[idx], y_train_bal.iloc[idx])

X_train_nca = nca.transform(X_train_sc).astype(np.float32)
X_test_nca = nca.transform(X_test_sc).astype(np.float32)

# 3. Stack Features
X_train_final = np.hstack([X_train_sc, X_train_nca])
X_test_final = np.hstack([X_test_sc, X_test_nca])

print(f"ResNet Feature Dimension: {X_train_final.shape[1]}")

--- Engineering Vector Space (ResNet) ---
ResNet Feature Dimension: 57


## 5. Training Generalist ResNet (Focal Loss)

In [6]:
print("--- Training Generalist ResNet ---")

# Focal Loss (Standard setup)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# ResNet Model
class ResNetClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# Train
X_t = torch.from_numpy(X_train_final)
y_t = torch.from_numpy(y_train_bal.values).long()
train_dl = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

classes = np.unique(y_train_bal)
w = compute_class_weight('balanced', classes=classes, y=y_train_bal)
w_t = torch.tensor(w, dtype=torch.float32).to(device)

resnet = ResNetClassifier(X_train_final.shape[1], len(classes)).to(device)
opt = optim.AdamW(resnet.parameters(), lr=0.001)
crit = FocalLoss(alpha=w_t)

for epoch in range(40):
    resnet.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(resnet(xb), yb)
        loss.backward()
        opt.step()
print("ResNet Trained.")

--- Training Generalist ResNet ---
ResNet Trained.


In [7]:
# Improved Hybrid Logic: Logit Boosting
def predict_boosted(resnet_model, sniper_model, X_vec, X_raw, encoder, attack_map):
    resnet_model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X_vec)), batch_size=1024, shuffle=False)
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            all_logits.append(resnet_model(batch[0].to(device)).cpu().numpy())
    logits = np.concatenate(all_logits)
    
    # Sniper Probability
    sniper_probs = sniper_model.predict_proba(X_raw)[:, 1]
    
    # Identify indices of R2L/U2R classes in the encoder
    # FIX: Encoder classes are ALREADY categories (dos, r2l), no need to map via attack_map
    rare_indices = []
    for idx, cls in enumerate(encoder.classes_):
        if cls in ['r2l', 'u2r']:
            rare_indices.append(idx)
            
    # Boost: If Sniper says "High Probability", boost logits of ALL R2L classes
    for i, prob in enumerate(sniper_probs):
        if prob > 0.5: # Sniper thinks it's Rare
            logits[i, rare_indices] += 5.0 # Massive boost to Rare logits
            
    final_idx = np.argmax(logits, axis=1)
    return encoder.inverse_transform(final_idx)

# --- Validation ---
print("--- Loading NSL-KDD ---")
NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]
try:
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')

    X_nsl = df_nsl.drop(['label', 'category', 'difficulty_level'], axis=1, errors='ignore')
    y_nsl = df_nsl['category']

    # Prepare Features
    X_nsl_eng, _ = prepare_features(X_nsl, fit=False, encoders=encoders)
    X_nsl_sc = scaler.transform(X_nsl_eng).astype(np.float32)
    X_nsl_nca = nca.transform(X_nsl_sc).astype(np.float32)
    X_nsl_final = np.hstack([X_nsl_sc, X_nsl_nca])

    # Predict
    y_pred_cat = predict_boosted(resnet, sniper, X_nsl_final, X_nsl_eng, le_y, attack_map)
    
    print(f"\n>>> HYBRID SNIPER ACCURACY: {accuracy_score(y_nsl, y_pred_cat):.2%}")
    print(classification_report(y_nsl, y_pred_cat))
except Exception as e:
    print(f"Error loading from URL: {e}")

--- Loading NSL-KDD ---

>>> HYBRID SNIPER ACCURACY: 75.98%
              precision    recall  f1-score   support

         dos       0.96      0.71      0.82      7636
      normal       0.72      0.96      0.83      9711
       probe       0.61      0.65      0.63      2423
         r2l       0.68      0.29      0.40      2574
         u2r       0.15      0.23      0.18       200

    accuracy                           0.76     22544
   macro avg       0.62      0.57      0.57     22544
weighted avg       0.78      0.76      0.75     22544

